# Where New York Goes for Indian

DBSCAN on every active restaurant the health department lists as Indian. Run `fetch_week1.py day4` first.

## 1 · Settings

In [ ]:
DAY      = 4
PROMPT   = "Clusters"
SLUG     = "clusters-nyc-indian"
NOTEBOOK = "day04_clusters_nyc_indian.ipynb"
TITLE    = "Where New York\nGoes for Indian"
CREDITS  = "Restaurants: NYC Department of Health restaurant inspections, active restaurants listed as Indian. Boundary: US Census Bureau."

## 2 · Load

In [ ]:
import geopandas as gpd, matplotlib.pyplot as plt, numpy as np, pandas as pd
from sklearn.cluster import DBSCAN
from mapkit import poster, publish
from mapkit.paths import PROCESSED

food = gpd.read_file(PROCESSED / "nyc_indian.geojson")
city = gpd.read_file(PROCESSED / "places_six.geojson").query("city == 'nyc'")

## 3 · Analysis, every number on the map comes from here

In [ ]:
EPS_M, MIN_SIZE = 400, 5      # at least five restaurants, each about a five-minute walk from the next
m = food.to_crs(32618)
food["cluster"] = DBSCAN(eps=EPS_M, min_samples=MIN_SIZE).fit_predict(np.c_[m.geometry.x, m.geometry.y])
sizes = food[food.cluster >= 0].groupby("cluster").size().sort_values(ascending=False)
in_clusters = int(sizes.sum())

# Names by where each cluster's center falls, checked by hand against its street addresses
PLACES = {"Curry Hill": (-73.986, 40.745), "Midtown West": (-73.986, 40.762), "Midtown East": (-73.972, 40.755), "East Village": (-73.985, 40.727),
          "Greenwich Village": (-74.001, 40.730), "Tribeca": (-74.008, 40.716), "Upper West Side": (-73.966, 40.801),
          "Richmond Hill": (-73.829, 40.689), "Richmond Hill south": (-73.828, 40.694), "Jackson Heights": (-73.892, 40.749), "Bedford-Stuyvesant": (-73.951, 40.681), "Floral Park": (-73.711, 40.737)}
centers = food[food.cluster >= 0].groupby("cluster").geometry.apply(lambda g: (g.x.mean(), g.y.mean()))
names = {c: min(PLACES, key=lambda p: (PLACES[p][0] - xy[0]) ** 2 + (PLACES[p][1] - xy[1]) ** 2) for c, xy in centers.items()}
assert len(set(names.values())) == len(names), "two clusters got the same name"
print(f"Indian restaurants  {len(food)}")
print(f"Clusters            {len(sizes)}")
print(f"In a cluster        {in_clusters} ({in_clusters / len(food) * 100:.0f}%)")
print(f"Dosa in the name    {int(food.dosa_named.astype(bool).sum())}")
for c, n in sizes.items():
    print(f"  {names[c]:22} {n}")

## 4 · Map

In [ ]:
SAFFRON, LAND, CHILI, INK = "#f2ae21", "#fff3d6", "#b3201f", "#3a120c"
fig, ax = poster.page(SAFFRON)
city.to_crs(3857).plot(ax=ax, color=LAND, edgecolor=INK, lw=0.9, zorder=1)
food_m = food.to_crs(3857)
ax.set_axis_off()
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -74.215, 40.43, -73.69, 40.925)
W = fx1 - fx0

# ripples around each cluster, thinning as they spread
ranked = list(sizes.index)
for c in ranked:
    pts = food_m[food_m.cluster == c]
    hull = pts.geometry.union_all().convex_hull
    for grow, lw, alpha in [(150, 1.3, 1), (340, 0.75, 0.85), (580, 0.42, 0.7), (880, 0.22, 0.55)]:
        gpd.GeoSeries([hull.buffer(grow * 1.32)], crs=3857).plot(ax=ax, facecolor="none", edgecolor=CHILI, lw=lw, alpha=alpha, zorder=3)
    gpd.GeoSeries([hull.buffer(150 * 1.32)], crs=3857).plot(ax=ax, facecolor=CHILI, alpha=0.2, lw=0, zorder=2)
food_m[food_m.cluster < 0].plot(ax=ax, color=INK, markersize=9, alpha=0.55, lw=0, zorder=4)
food_m[food_m.cluster >= 0].plot(ax=ax, color=CHILI, markersize=19, lw=0, zorder=5)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# a numbered marker on each cluster, explained in the list below
OFFSET = {"Midtown West": (-0.018, 0.012), "Midtown East": (0.022, 0.012), "Curry Hill": (-0.026, -0.004), "East Village": (0.026, -0.006),
          "Greenwich Village": (-0.024, -0.002), "Tribeca": (-0.024, -0.008), "Richmond Hill south": (0.024, 0.016), "Richmond Hill": (-0.024, -0.012)}
for rank, c in enumerate(ranked, 1):
    cx, cy = food_m[food_m.cluster == c].geometry.union_all().centroid.coords[0]
    dx, dy = OFFSET.get(names[c], (0.024, 0.016))
    ax.text(cx + W * dx, cy + W * dy, str(rank), color="white", fontproperties=poster.font(poster.POPPINS_BOLD, 9.5), ha="center", va="center", zorder=8,
            bbox=dict(boxstyle="circle,pad=0.28", fc=INK, ec="none"))

fig.text(0.06, 0.965, "Where New York", color=INK, fontproperties=poster.font(poster.YATRA, 38), va="top")
fig.text(0.06, 0.917, "Goes for Indian", color=CHILI, fontproperties=poster.font(poster.YATRA, 38), va="top")
fig.text(0.06, 0.862, f"{len(food)} Indian restaurants. {in_clusters} sit in {len(sizes)}\ntight clusters of five or more, each a\nfive-minute walk from the next.", color=INK,
         fontproperties=poster.font(poster.POPPINS, 12.5), va="top", linespacing=1.4)

# the list along the bottom
for i, c in enumerate(ranked):
    col, row = i // 4, i % 4
    x, y = 0.06 + col * 0.31, 0.145 - row * 0.022
    fig.text(x, y, f"{i + 1}", color=CHILI, fontproperties=poster.font(poster.POPPINS_BOLD, 11), va="center")
    fig.text(x + 0.026, y, f"{names[c].replace(' south', ' south')}  {sizes[c]}", color=INK, fontproperties=poster.font(poster.POPPINS, 11), va="center")
poster.credit(fig, CREDITS, path=poster.POPPINS, name_path=poster.POPPINS_BOLD, color="#5e3320", name_color=INK, size=8.4)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{len(food)} Indian restaurants, {len(sizes)} clusters", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)